# PCG Convergence Study  
Using NGSolve's built-in pcg solver and our finite element space hierarchy  


In [1]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
import time
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src import multigrid_cycles
from multigrid_cycles import *
from src import NGSolve_utils
from NGSolve_utils import *

In [2]:
# Getting colors for plots
cmap = plt.get_cmap('twilight_shifted')
cmap_nums = [tuple(cmap(x)[:3]) for x in np.linspace(0, 1,)]
# Setting options for plots
draw_opts = dict(
    Fullscreen=True,
    deformation=True,
    colors=cmap_nums,
    radius=0.75,
    center=[0.5, 0.5, 0.5],
    settings={
        "Objects": {"Wireframe":False},
        "camera": {
            "transformations":[
                {"type": "rotateX", "angle": -45}
            ]
        },
    },
)


In [3]:
# Setting up the problem we will solve on every level

# Boundary conditions
DIRICHLET = "left|right"

# LHS bilinear form
def poisson_bilinear(a, u, v):
    a += InnerProduct(grad(u), grad(v)) * dx

# RHS linear form
rhs_cf = 0
def poisson_linear(f, u, v):
    f += rhs_cf * v * dx

# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))


In [4]:

# Call our setup function to put these together
poisson_setup = build_form_setup(bilinear=poisson_bilinear, linear=poisson_linear)

# Define the coarsest mesh for our hierarchy of many sized meshes
N = 16
coarsest_mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
parfait = build_hierarchy(
    coarsest_mesh,
    poisson_setup,
    n_refines=5,
    order=1,
    dirichlet=DIRICHLET,
    dirichlet_value={"left": 0.0, "right":0.0},
    verbose=True,
)


  lev     ndof              A       P(c->f)    nfree  nfixed
  ---  -------  -------------  ------------  -------  ------
    0      339        339x339             -      305      34  (coarse)
    1     1289      1289x1289      1289x339     1223      66
    2     5025      5025x5025     5025x1289     4895     130
    3    19841    19841x19841    19841x5025    19583     258
    4    78849    78849x78849   78849x19841    78335     514
    5   314369  314369x314369  314369x78849   313343    1026  (fine)


In [5]:
finest = parfait.finest
finest.set_initial_guess(x0)

initial_plot = Draw(
        finest.gfu,
        finest.mesh,
        "initial guess (finest)",
        **draw_opts,
        )

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

In [6]:
x = finest.gfu.vec
b = finest.f.vec
finest.enforce_dirichlet(x)
print("residual norm before solve",finest.residual_norm(b,x))
finest.coarse_solve(b,x)
print("residual norm after solve",finest.residual_norm())

residual norm before solve 0.27664934473288183
residual norm after solve 0.0


In [7]:
finest.set_initial_guess(x0)
restart_plot = Draw(
        finest.gfu,
        finest.mesh,
        "going back to initial guess (finest)",
        **draw_opts,
        )

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

In [8]:
c = Preconditioner(finest.a, "local")
c.Update()
x2 = finest.gfu.vec
print("residual norm before CG", finest.residual_norm(b,x2))
solvers.BVP(bf=finest.a, lf=finest.f, gf=finest.gfu, pre=c, maxsteps=500, tol=1e-10, print=False)
cg_sol = finest.gfu.vec
print("residual norm after CG", finest.residual_norm(b,cg_sol))
cg_plot = Draw(finest.gfu, finest.mesh, "solution from CG", **draw_opts)

residual norm before CG 0.27664934473288183
residual norm after CG 0.09921179699601006


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

In [9]:
level_norms = []
for lvl in reversed(parfait.levels):
    level = lvl
    print("This level has ", level.ndof, " degrees of freedom")
    level.set_initial_guess(x0)
    blevel = level.f.vec
    initial_plot = Draw(
        level.gfu,
        level.mesh,
        **draw_opts,
        )
    clevel = Preconditioner(level.a, "local")
    clevel.Update()
    xl = level.gfu.vec
    print("Before solve, the residual norm is ", level.residual_norm(blevel,xl))
    solvers.BVP(bf=level.a, lf=level.f, gf=level.gfu, pre=clevel, maxsteps=500, tol=1e-10, print=True)
    cg_sol = level.gfu.vec
    level_norms.append(level.residual_norm(blevel, cg_sol))
    print("After CG, the residual norm is ", level_norms[-1])
    solved_plot = Draw(level.gfu, level.mesh, **draw_opts)

print("Here are the errors after CG for all the levels:")
for l in level_norms: print(l, '\n')



This level has  314369  degrees of freedom


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

Before solve, the residual norm is  0.27664934473288183
CG iteration 1, residual = 0.16056449435862805     
CG iteration 2, residual = 0.1955969455711357     
CG iteration 3, residual = 0.2233591717095805     
CG iteration 4, residual = 0.2861862761855445     
CG iteration 5, residual = 0.36439192862071235     
CG iteration 6, residual = 0.3673474855240257     
CG iteration 7, residual = 0.42752737619790093     
CG iteration 8, residual = 0.4298121309310005     
CG iteration 9, residual = 0.4347199495087951     
CG iteration 10, residual = 0.4327132715782225     
CG iteration 11, residual = 0.4258458903040004     
CG iteration 12, residual = 0.3935368331506342     
CG iteration 13, residual = 0.3896088523795858     
CG iteration 14, residual = 0.35569638941757975     
CG iteration 15, residual = 0.33484793460323264     
CG iteration 16, residual = 0.3148225099493125     
CG iteration 17, residual = 0.29151835931758624     
CG iteration 18, residual = 0.268095782343669     
CG iteration

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

This level has  78849  degrees of freedom


...

This level has  339  degrees of freedom


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

Before solve, the residual norm is  4.073610896107988
CG iteration 1, residual = 2.220805689144803     
CG iteration 2, residual = 0.5108602647014844     
CG iteration 3, residual = 0.6015021639008963     
CG iteration 4, residual = 0.588933947541298     
CG iteration 5, residual = 0.47091156103317244     
CG iteration 6, residual = 0.3798329132193529     
CG iteration 7, residual = 0.286370822561482     
CG iteration 8, residual = 0.24762445787830004     
CG iteration 9, residual = 0.21452819982371696     
CG iteration 10, residual = 0.21015121304734202     
CG iteration 11, residual = 0.22286697433406286     
CG iteration 12, residual = 0.32876086501324614     
CG iteration 13, residual = 0.29628798716278787     
CG iteration 14, residual = 0.11697107664248667     
CG iteration 15, residual = 0.05798443335390604     
CG iteration 16, residual = 0.042824232481968726     
CG iteration 17, residual = 0.029467174554985057     
CG iteration 18, residual = 0.02186629536419786     
CG itera

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

Here are the errors after CG for all the levels:
0.09921179699601006 

0.0015565852409851276 

9.59860940357241e-07 

1.6025362216059435e-10 

2.7759627698871177e-10 

3.128894836424803e-10 

